In [ ]:
import tensorly as tl


In [ ]:
import cvxpy as cp
import numpy as np

# Ensure repeatably random problem data.
np.random.seed(0)

# Generate random data matrix A.
m = 10
n = 10
o = 5
k = 2
D = 10*np.ones((m, n, o)) + np.random.randn(m, n, o)

# Initialize Y randomly.
A_init = 10*np.ones((m, k))
B_init = 10*np.ones((n, k))
C_init = 10*np.ones((o, k))




In [ ]:
Pred_A = np.einsum('ir, jr, kr ->ijk', A_init, B_init, C_init)

In [ ]:
# Ensure same initial random Y, rather than generate new one
# when executing this cell.
B = B_init
C = C_init

# Perform alternating minimization.
MAX_ITERS = 100
residual = np.zeros(MAX_ITERS)
for iter_num in range(0, 1+MAX_ITERS):

    if iter_num % 3 == 0:
        A = cp.Variable(shape=(n, k))
        constraint = [A >= 0]
        prediction = A@tl.tenalg.khatri_rao([C, B]).T
    elif iter_num % 3 == 1:
        B = cp.Variable(shape=(m, k))
        constraint = [B >= 0]
        prediction = B@tl.tenalg.khatri_rao([A, C]).T
    elif iter_num % 3 == 2:
        C = cp.Variable(shape=(o, k))
        constraint = [C >= 0]
        prediction = C@tl.tenalg.khatri_rao([B, A]).T

    obj = cp.Minimize(cp.norm(D.reshape(prediction.shape) - prediction, 'fro')/D.size)
    prob = cp.Problem(obj, constraint)
    prob.solve(solver=cp.SCS, max_iters=10000)

    if prob.status != cp.OPTIMAL:
        raise Exception("Solver did not converge!")

    print('Iteration {}, residual norm {}'.format(iter_num, prob.value))
    residual[iter_num-1] = prob.value

    # Convert variable to NumPy array constant for next iteration.
    if iter_num % 3 == 0:
        A = A.value
    elif iter_num%3 == 1:
        B = B.value
    else:
        C = C.value

In [ ]:
A

In [ ]:
B

In [ ]:
C

In [ ]:
np.einsum('ir, jr, kr ->ijk', A, B, C)